# NIH Chest X-ray - Model Evaluation

This notebook evaluates the models. The resulting file is stored.

## Contents:
1. Environment Setup & Imports
2. Experiment Configuration & Reproducibility
3. Device Configuration
4. Dataset Loading
5. Model Loading
6. Inference on Test Set
7. Threshold Calibration
8. Prediction Binarization
9. Model Evaluation
10. Results Saving
11. Summary Metrics

## 1. Imports & Setup

This section imports required libraries and evaluation utilities. It ensures reproducibility and loads the trained models for inference and evaluation.

In [16]:
import sys
import torch
import numpy as np
import json
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

import config
from dataset import get_dataloaders
from models import build_model
from evaluate import (
    collect_predictions,
    apply_thresholds,
    calibrate_thresholds,
    evaluate_model
)
from utils import seed_everything

## 2. Reproducibility & Experiment Selection

This section sets the experiment name and ensures reproducible evaluation conditions.

In [17]:
seed_everything()

EXPERIMENT_NAME = "exp_01_densenet_baseline"
config.set_experiment(EXPERIMENT_NAME)

print("Experiment:", config.EXPERIMENT_NAME)

Experiment: exp_01_densenet_baseline


## 3. Device Setup

This section selects the computation device used for evaluation.

In [18]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

device

device(type='mps')

## 4. Load Dataset

This section loads the test DataLoader using the same preprocessing pipeline as training.

In [15]:
train_loader, val_loader, test_loader = get_dataloaders()

print("Test samples:", len(test_loader.dataset))

[dataset] Loading metadata ...
[dataset] Subset mode: 1002 train+val, 202 test images
[dataset] Train/val patient overlap: 0
[dataset] Split sizes - train: 894, val: 108, test: 202
Test samples: 202


## 5. Load Trained Model

This section loads the trained model checkpoint that will be evaluated on unseen test data.

In [19]:
model_name = "densenet121"

model = build_model(model_name, pretrained=False).to(device)

ckpt_path = config.CHECKPOINT_DIR / f"{model_name}_best.pt"
model.load_state_dict(torch.load(ckpt_path, map_location=device))

model.eval()

print("Loaded:", ckpt_path)

Loaded: /Users/vesco/Documents/Projects/xray-master/outputs/exp_01_densenet_baseline/checkpoints/densenet121_best.pt


## 6. Run Inference (Test Set Predictions)

This section runs inference on the full test set and collects raw probabilities and ground truth labels.

In [10]:
labels, probs = collect_predictions(model, test_loader, device)

print("Labels shape:", labels.shape)
print("Probs shape:", probs.shape)

print("Sample labels:", labels[:5])
print("Sample probabilities:", probs[:5])

Labels shape: (202, 15)
Probs shape: (202, 15)
Sample labels: [[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]
Sample probabilities: [[0.39288628 0.3975461  0.2769274  0.2594647  0.34494987 0.3106195
  0.4085719  0.22596575 0.3896067  0.26755014 0.41690817 0.32918325
  0.2095639  0.3772002  0.7298113 ]
 [0.50724036 0.3321876  0.44106305 0.30904227 0.44272378 0.2568469
  0.2970843  0.22962563 0.49667782 0.25033683 0.4081575  0.3682124
  0.25767142 0.34393054 0.7298577 ]
 [0.5946406  0.3376927  0.47047505 0.39773402 0.5211947  0.28633064
  0.29564843 0.26708135 0.5335759  0.34651163 0.41671938 0.36643422
  0.2648182  0.40955028 0.6361555 ]
 [0.3818599  0.35577154 0.3933941  0.35400516 0.33940545 0.2872286
  0.24905066 0.27321684 0.43302402 0.36419502 0.38957804 0.26912645
  0.2237699  0.3239754  0.6

## 7. Threshold Calibration (Validation Set)

This section calibrates optimal classification thresholds per class using the validation set to improve decision-making in imbalanced multi-label classification.

In [22]:
thresholds = calibrate_thresholds(model, val_loader, model_name)
thresholds

[evaluate] Calibrating thresholds on the validation set (108 samples) ...
[evaluate] Thresholds saved -> /Users/vesco/Documents/Projects/xray-master/outputs/exp_01_densenet_baseline/results/densenet121_thresholds.json


array([0.53, 0.5 , 0.05, 0.36, 0.49, 0.5 , 0.5 , 0.5 , 0.42, 0.5 , 0.32,
       0.5 , 0.23, 0.5 , 0.57], dtype=float32)

## 8. Apply Thresholds to Test Set

This section converts probability outputs into binary predictions using calibrated thresholds.

In [23]:
preds = apply_thresholds(probs, thresholds)

print("Preds shape:", preds.shape)

Preds shape: (202, 15)


## 9. Full Model Evaluation

This section computes comprehensive evaluation metrics including AUROC, AUPRC, precision, recall, F1-score, calibration metrics, and per-class performance.

In [24]:
results = evaluate_model(
    model=model,
    test_loader=test_loader,
    model_name=model_name,
    thresholds=thresholds
)

[evaluate] Running inference on the test set (202 samples) ...
[evaluate] Computing calibrated metrics ...

  Evaluation Results - densenet121
Class                     Thr    AUROC    AUPRC    Prec  Recall      F1  Support
--------------------------------------------------------------------------------
Atelectasis              0.53   0.6689   0.2382  0.2295  0.5000  0.3146       28
Cardiomegaly             0.50   0.4129   0.0220  0.0000  0.0000  0.0000        4
Consolidation            0.05   0.6043   0.2355  0.0941  1.0000  0.1719       19
Edema                    0.36   0.7366   0.1924  0.1519  0.7500  0.2526       16
Effusion                 0.49   0.5905   0.1658  0.1429  0.4074  0.2115       27
Emphysema                0.50      N/A      N/A  0.0000  0.0000  0.0000        0
Fibrosis                 0.50   0.6825   0.0225  0.0000  0.0000  0.0000        2
Hernia                   0.50      N/A      N/A  0.0000  0.0000  0.0000        0
Infiltration             0.42   0.6214   0.4790

## 10. Save Evaluation Summary

This section saves evaluation results in JSON format for later comparison across models and experiments.

In [25]:
summary_path = config.RESULTS_DIR / f"{model_name}_evaluation.json"

with open(summary_path, "w") as f:
    json.dump(results, f, indent=2)

print("Saved:", summary_path)

Saved: /Users/vesco/Documents/Projects/xray-master/outputs/exp_01_densenet_baseline/results/densenet121_evaluation.json


## 11. Top Metrics

This section prints key summary metrics for quick interpretation of model performance.

In [26]:
print("Macro AUROC:", results["macro"]["auroc"])
print("Macro AUPRC:", results["macro"]["auprc"])
print("Macro F1:", results["macro"]["f1"])
print("Hamming Loss:", results["macro"]["hamming_loss"])

Macro AUROC: 0.591
Macro AUPRC: 0.1906
Macro F1: 0.1479
Hamming Loss: 0.3168
